In [1]:
# from huggingface_hub import snapshot_download

# snapshot_download(repo_id="anhnh2002/vnTTS",
#                   repo_type="model",
#                   local_dir="model/")

In [ ]:
# !pip install cutlet

In [ ]:
# pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126

In [2]:
#check cuda
import torch
if torch.cuda.is_available():
    print("CUDA is available.")
    print("CUDA version:", torch.version.cuda)
    print("PyTorch version:", torch.__version__)
    print("Number of GPUs:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())

CUDA is available.
CUDA version: 11.8
PyTorch version: 2.7.1+cu118
Number of GPUs: 1
Current device: 0


In [1]:
from pprint import pprint
import torch
import torchaudio
from tqdm import tqdm
from underthesea import sent_tokenize
from vinorm import TTSnorm
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

device = "cuda:0"

xtts_checkpoint = "model/model.pth"
xtts_config = "model/config.json"
xtts_vocab = "model/vocab.json"

config = XttsConfig()
config.load_json(xtts_config)
XTTS_MODEL = Xtts.init_from_config(config)
XTTS_MODEL.load_checkpoint(config,
                            checkpoint_path=xtts_checkpoint,
                            vocab_path=xtts_vocab,
                            use_deepspeed=False)
XTTS_MODEL.to(device)
# print(next(XTTS_MODEL.parameters()).device)

/mnt/d/Ky 4/vietnamese-speech-chatbox/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an excepti

Xtts(
  (gpt): GPT(
    (conditioning_encoder): ConditioningEncoder(
      (init): Conv1d(80, 1024, kernel_size=(1,), stride=(1,))
      (attn): Sequential(
        (0): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Identity()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (1): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Identity()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (2): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Ide

In [56]:
def preprocess_text(text, language="vi"):
    if language == "vi":
        text = TTSnorm(text, unknown=False, lower=False, rule=True)
    
    # split text into sentences
    if language in ["ja", "zh-cn"]:
        sentences = text.split("。")
    else:
        sentences = sent_tokenize(text)

    chunks = []
    chunk_i = ""
    len_chunk_i = 0
    for sentence in sentences:
        chunk_i += " " + sentence
        len_chunk_i += len(sentence.split())
        if len_chunk_i > 30:
            chunks.append(chunk_i.strip())
            chunk_i = ""
            len_chunk_i = 0

    if (len(chunks) > 0) and (len_chunk_i < 15):
        chunks[-1] += chunk_i
    else:
        chunks.append(chunk_i)

    return chunks

In [ ]:
speaker_audio_file = "model/samples/nu-luu-loat.wav"
# speaker_audio_file = "model/vi_man.wav"
# speaker_audio_file ="model/nu-luu-loat.wav"

gpt_cond_latent, speaker_embedding = XTTS_MODEL.get_conditioning_latents(
    audio_path=speaker_audio_file,
    gpt_cond_len=XTTS_MODEL.config.gpt_cond_len,
    max_ref_length=XTTS_MODEL.config.max_ref_len,
    sound_norm_refs=XTTS_MODEL.config.sound_norm_refs,
)

print(gpt_cond_latent)
print(speaker_embedding)

# torch.save()

tensor([[[ 0.6907,  0.7729,  0.4243,  ...,  1.1100, -0.0158, -0.4993],
         [-0.0752,  0.0320,  0.7642,  ...,  1.4668,  0.5185, -0.0191],
         [ 0.4739,  0.1560, -0.7226,  ...,  1.0538, -0.6100, -0.9636],
         ...,
         [ 0.0864,  1.6816, -0.8442,  ...,  0.3323,  0.5421,  0.3424],
         [-0.2773,  1.1737,  0.5766,  ...,  0.2312, -0.4370, -0.4469],
         [-0.6062,  0.9037,  1.0814,  ...,  0.4370, -1.0872, -0.0135]]],
       device='cuda:0')
tensor([[[ 5.0528e-03],
         [ 1.1395e-02],
         [-1.4524e-02],
         [ 3.8005e-02],
         [ 4.5182e-03],
         [ 2.3954e-02],
         [-2.8330e-02],
         [ 1.1783e-02],
         [ 5.7613e-02],
         [ 6.3478e-02],
         [ 1.2202e-02],
         [-4.2736e-02],
         [ 3.1681e-02],
         [ 8.4034e-02],
         [-1.1321e-02],
         [-7.3634e-02],
         [ 2.0425e-03],
         [-5.2167e-02],
         [ 1.0635e-02],
         [-3.4519e-02],
         [-3.8537e-02],
         [-4.8609e-02],
      

In [5]:
def tts(
    model: Xtts,
    text: str,
    language: str,
    gpt_cond_latent: torch.Tensor,
    speaker_embedding: torch.Tensor,
    verbose: bool = False,
):
    # preprocess text
    chunks = preprocess_text(text, language)

    wav_chunks = []
    for text in tqdm(chunks):
        if text.strip() == "":
            continue
        wav_chunk = model.inference(
            text=text,
            language=language,
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=speaker_embedding,
            length_penalty=1.0,
            repetition_penalty=10.0,
            top_k=10,
            top_p=0.5,
        )

        wav_chunk["wav"] = torch.tensor(wav_chunk["wav"])

        wav_chunks.append(wav_chunk["wav"])

    out_wav = torch.cat(wav_chunks, dim=0).unsqueeze(0).cpu()

    return out_wav

from IPython.display import Audio

audio = tts(
    model=XTTS_MODEL,
    # text="Xin chào, tôi là một hệ thống chuyển đổi văn bản tiếng Việt thành giọng nói. Hello, I am a Vietnamese text to speech conversion system.", 
    # text = """ID giáo trình: 11845
    #         Tên giáo trình: Communication and In-Group Working Skills Kỹ năng giao tiếp và cộng tác
    #         Mã chủ đề: SSG104
    #         Số tín chỉ: 3
    #         Cấp độ đào tạo: Cử nhân
    #         Phân bổ thời gian: Tổng thời lượng học là 150 giờ, bao gồm 45 giờ học trực tiếp tương đương với 60 phiên học, 0.5 giờ dành cho kỳ thi cuối kỳ, và 104.5 giờ tự học.
    #         Điều kiện tiên quyết: Không có
    # """,
    text = "Tất cả quy trình diễn ra tự động, trực quan, thân thiện""",
    language="vi",
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    verbose=True,   
)

Audio(audio, rate=24000)

100%|██████████| 1/1 [00:10<00:00, 10.98s/it]


In [59]:
text = """
Một ngày nọ, cậu cháu trai của ông Nam – bé Minh – tò mò leo lên gác và phát hiện ra chiếc đồng hồ. “Ông ơi, sao đồng hồ không kêu nữa ạ?” – Minh hỏi với đôi mắt tròn xoe. Ông Nam nhìn cháu, mỉm cười hiền hậu rồi nói: “Vì không ai còn lắng nghe tiếng của nó nữa con ạ.” Tối hôm đó, ông Nam quyết định mang chiếc đồng hồ xuống, cẩn thận lau chùi, tra dầu và điều chỉnh từng bánh răng một cách tỉ mỉ.

Sau gần ba tiếng đồng hồ, chiếc đồng hồ cổ bỗng kêu “tích tắc” lần đầu tiên sau hơn hai mươi năm im lặng. Minh reo lên vui sướng, còn ông Nam thì rưng rưng nước mắt. Tiếng tích tắc nhỏ bé ấy như mang theo hơi thở của quá khứ, gắn kết những thế hệ trong gia đình bằng một sợi dây vô hình nhưng bền chặt.

Từ ngày hôm đó, mỗi buổi tối, cả gia đình lại ngồi quây quần trong phòng khách, lắng nghe tiếng đồng hồ như một bản giao hưởng thầm lặng của ký ức – một minh chứng rằng đôi khi, những điều xưa cũ nhất lại là thứ kết nối con người sâu sắc nhất.
"""

len(chunk[i]) ≥ len(chunk[i+1]) ∀ i.

In [64]:
import time 
from IPython.display import display, Audio

chunks = preprocess_text(text, language="vi")
print("⚡️ Bắt đầu inference streaming-like...")

for i, chunk in enumerate(chunks):
    t0 = time.time()
    if chunk.strip() == "":
        continue

    # Stream generation của mỗi chunk
    for wav_chunk in XTTS_MODEL.inference_stream(
        text=chunk,
        language="vi",
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        stream_chunk_size=20,
        overlap_wav_len=1024,
        temperature=1.0,
        length_penalty=1.0,
        repetition_penalty=10.0,
        top_k=10,
        top_p=0.5,
    ):
        print(f"\n▶️ Playing chunk {i + 1}/{len(chunks)}: {chunk[:]}...")
        wav_chunk = torch.tensor(wav_chunk) if not isinstance(wav_chunk, torch.Tensor) else wav_chunk
        wav_chunk = wav_chunk.cpu().numpy()
        display(Audio(wav_chunk, rate=24000))
        duration_sec = len(wav_chunk) / 24000
        time.sleep(duration_sec)

    print(f"⏱️ Time in {i} chunk: {time.time() - t0:.2f}s")

print("✅ Done streaming.")

⚡️ Bắt đầu inference streaming-like...


AttributeError: 'int' object has no attribute 'device'

KeyboardInterrupt: 